In [3]:
#export MODEL="Your model here"
export MODEL="unsloth/Qwen3.5-35B-A3B-GGUF:UD-IQ3_S"
# to set sudo sysctl iogpu.wired_limit_mb=14800. Needs to be set on termimal outside for pure GPU
sysctl iogpu.wired_limit_mb # to check

iogpu.wired_limit_mb: 14800


In [4]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.010 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [5]:
# Pure GPU
llama-bench -hf $MODEL -ngl 99 -t 8 -ub 128,256,512 -p 1000,2000 -n 256,512

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.008 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [ ]:
# From llama.cpp readme. 
# -b 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 
# so may not be applicable to mac mini

In [ ]:
# batched bench seems the closer to llama-server as judged from the logs, thats' why it is here
# llama-batched-bench -hf $MODEL \
#     -ngl 99 --threads 8 -fa on \
#     -c 16000 \
#     -npp 1000 -ntg 1000 -npl 1

In [ ]:
llama-fit-params -hf $MODEL -fitt 512 # fitt leaves that around of memory in MiB in the target devices

In [ ]:
llama-server -hf $MODEL \
    --threads-http 1 --fit on --threads 8 --no-mmproj --reasoning off \
    --no-warmup -ngl -1 -np 1 --host 0.0.0.0 # -np 1 parameter is important